<a href="https://colab.research.google.com/github/yeugeneu/IAT360-NLP-3-RAG-Chatbot/blob/main/financial_claim_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies

In [27]:
!pip install -U transformers sentence-transformers -q kagglehub[pandas-datasets]

# Data Pre-processing

## Load and Inspect the Dataset
Load the [Financial Q&A - 10k](https://www.kaggle.com/datasets/yousefsaeedian/financial-q-and-a-10k/data)  dataset and inspect its structure.

Download the dataset to your Google Drvie location

In [ ]:
import os
import shutil
import kagglehub

# Define the target Google Drive directory specified by you
target_dir = "/content/drive/MyDrive/IAT 360/NLP-Group3/dataset"

# Ensure the target directory exists
os.makedirs(target_dir, exist_ok=True)

# Get the cache/download path
cache_path = kagglehub.dataset_download("yousefsaeedian/financial-q-and-a-10k")
print("Source path:", cache_path)

# Copy the files from the cache path to your Google Drive
if os.path.exists(cache_path):
    for item in os.listdir(cache_path):
        s = os.path.join(cache_path, item)
        d = os.path.join(target_dir, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.copytree(s, d)
        else:
            shutil.copy2(s, d)
    print("Successfully copied dataset files to your Google Drive:", target_dir)
else:
    print("Source path does not exist.")

Using Colab cache for faster access to the 'financial-q-and-a-10k' dataset.
Source path: /kaggle/input/financial-q-and-a-10k
Successfully copied dataset files to your Google Drive: /content/drive/MyDrive/IAT 360/NLP-Group3/dataset


View the dataset

In [28]:
import pandas as pd

# Path to the dataset in Google Drive under the new 'dataset' directory
dataset_path = "/content/drive/MyDrive/IAT 360/NLP-Group3/dataset/Financial-QA-10k.csv"

# Load the dataset into a pandas DataFrame
df = pd.read_csv(dataset_path)

# Display basic information and the first few rows
print(df.info())
print("\nFirst 3 rows:")
display(df.head(3))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  6998 non-null   object
 1   answer    6998 non-null   object
 2   context   6999 non-null   object
 3   ticker    7000 non-null   object
 4   filing    7000 non-null   object
dtypes: object(5)
memory usage: 273.6+ KB
None

First 3 rows:


,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K


## Extract data needed

We will probably need the `context` and `ticker` columns only since the filing year is always `2023_10K`

In [ ]:
# Add code here to clean the data into a df

# Generate Document Embeddings

In [ ]:
# Embedding the context to LLM

# Load LLM

Login to huggingface

In [29]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [ ]:
!hf auth whoami

✓ Logged in
  user: yeugeneu33


In [30]:
# from transformers import pipeline

# llm = pipeline("text-generation", model="meta-llama/Meta-Llama-3-8B-Instruct")
from transformers import AutoTokenizer
import transformers
import torch

model = "meta-llama/Meta-Llama-3-8B-Instruct" # meta-llama/Meta-Llama-3-8B-Instruct

tokenizer = AutoTokenizer.from_pretrained(model, use_auth_token=True)

In [31]:
from transformers import pipeline

llm = pipeline(
    "text-generation",  # LLM task
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [32]:
def get_llama_response(prompt: str) -> None:
    """
    Generate a response from the Llama model.

    Parameters:
        prompt (str): The user's input/question for the model.

    Returns:
        None: Prints the model's response.
    """
    sequences = llm(
        prompt,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=256,
    )
    print("Chatbot:", sequences[0]['generated_text'])

In [34]:
# Ask the model a question
question = "What significant invention did NVIDIA create in 1999?"
get_llama_response(question)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Chatbot: What significant invention did NVIDIA create in 1999? The answer is: the GeForce graphics processing unit (GPU). This invention revolutionized the gaming industry by providing a dedicated graphics processing unit that could handle complex graphics and 3D rendering tasks, leading to a significant improvement in gaming performance and visual quality. The GeForce GPU was a game-changer for the gaming industry, and it has continued to evolve and improve over the years, enabling the development of more sophisticated and immersive gaming experiences. Today, NVIDIA's GPUs are used not only in gaming but also in a wide range of applications, including artificial intelligence, deep learning, and professional visualization. So, the next time you play a game or watch a stunning visual effect, remember the significance of NVIDIA's GeForce GPU invention in 1999! #NVIDIA #GeForce #Gaming #GPU #Innovation #Technology #History #GamingIndustry #GraphicsProcessingUnit #ComputerGraphics #Artific